# MEL city bikes ETL

This notebook is part of data engineering project concentrated on building **ETL pipeline** using open datasets from the **detaMEL/Lille Metropole Open Data**.

---

## Table of Contents
1. Introduction
2. Environment Setup
3. Raw Data Profiling
4. Preliminary Observations
5. Next Steps

## 1. Introduction

This part of the project presents **the raw data extraction** and **initial data exploration (EDA)** which is the first phase of end-to-end **ETL pipeline** creation process based on the **MEL city bikes** datasets.

---

**Licence and Usage**

These datasets are licensed under a **Licence Ouverte Etalab 2.0**.
They were downloaded on **3 December 2025**.


**Reference**

- Ilévia - *Comptages vélos depuis 2013*, dataMEL, [link↗](https://data.lillemetropole.fr/catalogue/dataset/fb7cf2fb-9466-48eb-a141-3869a4c1860b)
- Ilévia -*Populations des communes et de la MEL*, dataMEL, [link↗](https://data.lillemetropole.fr/catalogue/dataset/1106c6dc-a5ec-4eb8-a4e0-9cc813a2aeda)
- Ilévia -*Communes et communes associées de la Métropole Européenne de Lille*, dataMEL, [link↗](https://data.lillemetropole.fr/catalogue/dataset/25e38054-3194-46ab-ab88-f2b144ef26ed)

---

**Objective**

The main objective iof this stage is to explore the raw datasets, assess data quality, detect structure patterns, and prepare documentation required for the data modeling and transformation stages.

This exploratory analysis supports later etaps: cleaning, schema design, database integration and ETL (Talend).

## 2. Environment Setup

In this section, the Python environment is initialized to enable working with the raw data. Firstly, **pandas** library is imported as the primary tool for loading, inspecting and manipulating structured data as part of the ETL workflow.

The following datasets are loaded into separate dataframes for for separate inspection:

- `velos_df` - spot counts of bicycles on the MEL cycle network using an electromagnetic detection system installed in the road surface at the stations,
- `populations_df` - municipal and metropolitan population for years 2006, 2015 ans 2020,
- `communes_df` - the municipalities of MEL.

This operation allows a clearer understanding of the structure, schema, column meanings, and potential data quality issues before designing the final data model.

In [3]:
import pandas as pd

In [7]:
velos_df = pd.read_csv("../data/raw/velos.csv")
populations_df = pd.read_csv("../data/raw/populations.csv")
communes_df = pd.read_csv("../data/raw/communes.csv")

## 3. Raw data profiling

### Bikes detection

The `velo_df` contains metadata on city bike usage detected by an electromagnetic system and aggregated into weekly intervals since 2013.

---

**Dataset characteristics:**

- **Rows:** 11,398
- **Columns:** 11
- **Data types:**
  - 4 numeric
  - 7 string/object
- **Missing values** detected in 1 column:
  - `code_insee`

---

**Key structural observations:**

- The `id` field appears to represent an electromagnetic sensor identifier.
- `code_insee` contains missing data for the coty of Villeneuve d Ascq.
- `nom` and `adresse` contain the same information.
- The `geom` column stores geometry values with string prefixes, which duplicate information already available in the `latitude` and `longitude` columns.

---

**Preliminary Implications and Required Actions:**

- Create a **dimension table** `dim_capteur`based on id, which will serve as the primary key and include geospatial attributes.
- Create a **dimension table** `dim_calendrier` to store time-related information.
- Keep identifiers (foreign keys) and aggregated sensor readings in the `fact_velos` **fact table**.
- Verify `code_insee` for Villeneuve d Ascq in `communes_df` and input the missing information, since this column is a likely foreign key candidate.
- Drop `nom` and `geom` as a duplicated information.

In [8]:
velos_df.head(3)

,id,nom,adresse,ville,code_insee,annee,semaine,mjo,longitude,latitude,geom
0,VEL012,Lille - Av. Le Corbusier,Lille - Av. Le Corbusier,LILLE,59350.0,2013,36,152.0,3.073626,50.638620,POINT (3.073625583618273 50.63862003542886)
1,VEL014,Lille - Grand Boulevard,Lille - Grand Boulevard,LILLE,59350.0,2013,36,156.0,3.087474,50.656063,POINT (3.087473889889443 50.6560630327743)
2,VEL012,Lille - Av. Le Corbusier,Lille - Av. Le Corbusier,LILLE,59350.0,2013,37,1973.0,3.073626,50.638620,POINT (3.073625583618273 50.63862003542886)


In [9]:
velos_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11398 entries, 0 to 11397
Data columns (total 11 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   id          11398 non-null  object 
 1   nom         11398 non-null  object 
 2   adresse     11398 non-null  object 
 3   ville       11398 non-null  object 
 4   code_insee  10824 non-null  float64
 5   annee       11398 non-null  int64  
 6   semaine     11398 non-null  int64  
 7   mjo         11398 non-null  float64
 8   longitude   11398 non-null  float64
 9   latitude    11398 non-null  float64
 10  geom        11398 non-null  object 
dtypes: float64(4), int64(2), object(5)
memory usage: 979.6+ KB


In [10]:
velos_df.describe(include = 'all')

,id,nom,adresse,ville,code_insee,annee,semaine,mjo,longitude,latitude,geom
count,11398,11398,11398,11398,10824.000000,11398.000000,11398.000000,11398.000000,11398.000000,11398.000000,11398
unique,37,37,37,24,NaN,NaN,NaN,NaN,NaN,NaN,37
top,VEL028,Lille - Rue Pierre Legrand,Lille - Rue Pierre Legrand,LILLE,NaN,NaN,NaN,NaN,NaN,NaN,POINT (3.082379600000004 50.63269003860525)
freq,505,505,505,2263,NaN,NaN,NaN,NaN,NaN,NaN,505
mean,NaN,NaN,NaN,NaN,59352.948725,2019.546324,25.801720,309.383050,3.068472,50.656754,NaN
std,NaN,NaN,NaN,NaN,169.107057,3.145712,15.204868,436.142823,0.085967,0.053521,NaN
min,NaN,NaN,NaN,NaN,59013.000000,2013.000000,1.000000,-8.000000,2.857053,50.544080,NaN
25%,NaN,NaN,NaN,NaN,59256.000000,2017.000000,13.000000,71.000000,3.017531,50.630528,NaN
50%,NaN,NaN,NaN,NaN,59350.000000,2020.000000,24.000000,133.000000,3.082380,50.646120,NaN
75%,NaN,NaN,NaN,NaN,59512.000000,2022.000000,39.000000,356.000000,3.131189,50.694153,NaN


In [11]:
velos_df.nunique()

id              37
nom             37
adresse         37
ville           24
code_insee      23
annee           12
semaine         53
mjo           1545
longitude       37
latitude        37
geom            37
dtype: int64

In [12]:
velos_df['nom'].equals(velos_df['adresse'])

True

In [15]:
velos_df_nan = velos_df[velos_df['code_insee'].isna()]
velos_df_nan.head()

,id,nom,adresse,ville,code_insee,annee,semaine,mjo,longitude,latitude,geom
37,VEL024,Villeneuve d Ascq - Bd Van Gogh,Villeneuve d Ascq - Bd Van Gogh,VILLENEUVE-D ASCQ,NaN,2013,48,27.0,3.131189,50.621346,POINT (3.131188998267782 50.621346028721256)
56,VEL024,Villeneuve d Ascq - Bd Van Gogh,Villeneuve d Ascq - Bd Van Gogh,VILLENEUVE-D ASCQ,NaN,2013,49,28.0,3.131189,50.621346,POINT (3.131188998267782 50.621346028721256)
76,VEL024,Villeneuve d Ascq - Bd Van Gogh,Villeneuve d Ascq - Bd Van Gogh,VILLENEUVE-D ASCQ,NaN,2013,50,23.0,3.131189,50.621346,POINT (3.131188998267782 50.621346028721256)
97,VEL024,Villeneuve d Ascq - Bd Van Gogh,Villeneuve d Ascq - Bd Van Gogh,VILLENEUVE-D ASCQ,NaN,2013,51,33.0,3.131189,50.621346,POINT (3.131188998267782 50.621346028721256)
119,VEL024,Villeneuve d Ascq - Bd Van Gogh,Villeneuve d Ascq - Bd Van Gogh,VILLENEUVE-D ASCQ,NaN,2014,2,34.0,3.131189,50.621346,POINT (3.131188998267782 50.621346028721256)


In [17]:
velos_df_nan.describe(include='all')

,id,nom,adresse,ville,code_insee,annee,semaine,mjo,longitude,latitude,geom
count,574,574,574,574,0.0,574.000000,574.000000,574.000000,574.000000,574.000000,574
unique,2,2,2,1,NaN,NaN,NaN,NaN,NaN,NaN,2
top,VEL024,Villeneuve d Ascq - Bd Van Gogh,Villeneuve d Ascq - Bd Van Gogh,VILLENEUVE-D ASCQ,NaN,NaN,NaN,NaN,NaN,NaN,POINT (3.131188998267782 50.621346028721256)
freq,400,400,400,574,NaN,NaN,NaN,NaN,NaN,NaN,400
mean,NaN,NaN,NaN,NaN,NaN,2020.054007,25.783972,197.278746,3.138525,50.625412,NaN
std,NaN,NaN,NaN,NaN,NaN,3.180253,15.269494,265.447705,0.011132,0.006171,NaN
min,NaN,NaN,NaN,NaN,NaN,2013.000000,1.000000,9.000000,3.131189,50.621346,NaN
25%,NaN,NaN,NaN,NaN,NaN,2017.000000,13.000000,32.000000,3.131189,50.621346,NaN
50%,NaN,NaN,NaN,NaN,NaN,2021.000000,24.000000,50.500000,3.131189,50.621346,NaN
75%,NaN,NaN,NaN,NaN,NaN,2023.000000,39.750000,321.250000,3.155389,50.634760,NaN


In [18]:
velos_df_nan['id'].value_counts()

VEL024    400
VEL026    174
Name: id, dtype: int64

In [19]:
velos_df_nan['adresse'].value_counts()

Villeneuve d Ascq - Bd Van Gogh    400
Villeneuve d Ascq - 8 Mai 1945     174
Name: adresse, dtype: int64

In [16]:
velos_df_nan.nunique()

id              2
nom             2
adresse         2
ville           1
code_insee      0
annee          12
semaine        53
mjo           239
longitude       2
latitude        2
geom            2
dtype: int64

### Population changes

The `populations_df` dataset contains information about changes in the number of inhabitants in municipal and metropolitan areas for the years 2006, 2015, and 2020.

---

**Dataset characteristics:**

- **Rows:** 98
- **Columns:** 13
- **Data types:**
  - 8 numeric
  - 5 string/object
- **Missing values** detected in 2 columns:
  - `epci`
  - `scot`

---

**Key structural observations:**

- `code_admin` is community administration identifier.
- `code_insee` is the municipality postal code (also present in `velos_df`).
- `epci` and `scot` contain only a single unique value each.
- Colums `pop_yyyy` and `evo_pop_mm_yy` represent population counts and population change across years.
- `surface_m2` and `geom` do not contain population-related information.

---
  
**Preliminary Implications and Required Actions:**

- Drop the `epci` and `scot`columns because they do not provide meaningful information.
- Drop `surface_m2` and `geom` as they are not directly related to population change and will be included in a future dimension table.

In [20]:
populations_df.head(3)

,code_adm,code_insee,nom,epci,scot,pop_2006,pop_2015,pop_2020,evo_pop_06_15,evo_pop_06_20,evo_pop_15_20,surface_m2,geom
0,COM_F59005,59005,Allennes-les-Marais,EPCI_245900410,SCOT_1000,3407,3413,3532,6,125,119,5.561176e+06,MULTIPOLYGON (((2.9630895928204106 50.52802738...
1,COM_F59009,59009,Villeneuve-d'Ascq,EPCI_245900410,SCOT_1000,61151,61920,61258,769,107,-662,2.757522e+07,MULTIPOLYGON (((3.1793022723575555 50.60930052...
2,COM_F59011,59011,Annœullin,EPCI_245900410,SCOT_1000,9802,10407,10679,605,877,272,9.008851e+06,MULTIPOLYGON (((2.8954464720510598 50.53189944...


In [22]:
populations_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 98 entries, 0 to 97
Data columns (total 13 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   code_adm       98 non-null     object 
 1   code_insee     98 non-null     int64  
 2   nom            98 non-null     object 
 3   epci           97 non-null     object 
 4   scot           97 non-null     object 
 5   pop_2006       98 non-null     int64  
 6   pop_2015       98 non-null     int64  
 7   pop_2020       98 non-null     int64  
 8   evo_pop_06_15  98 non-null     int64  
 9   evo_pop_06_20  98 non-null     int64  
 10  evo_pop_15_20  98 non-null     int64  
 11  surface_m2     98 non-null     float64
 12  geom           98 non-null     object 
dtypes: float64(1), int64(7), object(5)
memory usage: 10.1+ KB


In [23]:
populations_df.describe(include = 'all')

,code_adm,code_insee,nom,epci,scot,pop_2006,pop_2015,pop_2020,evo_pop_06_15,evo_pop_06_20,evo_pop_15_20,surface_m2,geom
count,98,98.000000,98,97,97,9.800000e+01,9.800000e+01,9.800000e+01,98.000000,98.000000,98.000000,9.800000e+01,98
unique,98,NaN,98,1,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,98
top,COM_F59005,NaN,Allennes-les-Marais,EPCI_245900410,SCOT_1000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,MULTIPOLYGON (((2.9630895928204106 50.52802738...
freq,1,NaN,1,97,97,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
mean,NaN,58743.387755,NaN,NaN,NaN,2.320131e+04,2.378996e+04,2.412755e+04,588.653061,926.244898,337.591837,1.379482e+07,NaN
std,NaN,5998.245669,NaN,NaN,NaN,1.160452e+05,1.189850e+05,1.206693e+05,3041.788968,4716.438362,1731.914386,6.774750e+07,NaN
min,NaN,0.000000,NaN,NaN,NaN,1.890000e+02,2.340000e+02,2.350000e+02,-1875.000000,-1837.000000,-694.000000,1.848695e+05,NaN
25%,NaN,59197.250000,NaN,NaN,NaN,1.595250e+03,1.678750e+03,1.818250e+03,-3.000000,17.000000,-65.000000,3.285191e+06,NaN
50%,NaN,59344.500000,NaN,NaN,NaN,4.223500e+03,4.713500e+03,4.801500e+03,171.500000,193.500000,47.000000,5.699581e+06,NaN
75%,NaN,59511.000000,NaN,NaN,NaN,1.254300e+04,1.300725e+04,1.301600e+04,457.500000,634.000000,212.250000,9.937928e+06,NaN


In [24]:
populations_df.nunique()

code_adm         98
code_insee       98
nom              98
epci              1
scot              1
pop_2006         96
pop_2015         98
pop_2020         98
evo_pop_06_15    89
evo_pop_06_20    94
evo_pop_15_20    94
surface_m2       98
geom             98
dtype: int64

### Municipalities details

The `communes_df` dataset provides administrative and geospatial information about municipalities.

---

**Dataset characteristics:**

- **Rows:** 97
- **Columns:** 30
- **Data types:**
  - 7 numeric
  - 21 string/object
- **Missing values** detected in 14 columns:
   - `code_mel`
   - `libelle_mel`
   - `code_deveco`
   - `libelle_deveco`
   - `code_enm`
   - `libelle_enm`
   - `code_m_locale`
   - `libelle_m_locale`
   - `code_gardo`
   - `libelle_gardo`
   - `code_dt`
   - `libelle_dt`
   - `code_gdt`
   - `libelle_gdt`

---
 
**Key structural observations:**

- The column `code_insee` is a unique municipality identifier and serves as a primary key referenced by foreign keys in the `velos_df` and `populations_df` fact tables.
- Several columns (e.g. `code_mel`, `libelle_mel`, `code_gardo`, `libelle_gardo`, `code_dt`) contain only one unique value. Missing values indicate flag **NA** for the category presented in the name of the column.
- Columns such as `code_gdt`, and `libelle_gdt` are empty.
---

**Preliminary Implications and Required Actions:**

- Input the value **59009** into the `code_insee` column for Villeneuve-d'Ascq in the `velos_df` fact table.
- Input **NA** for the object-type columns containing administrative flag information.
- Drop `code_gdt`, and `libelle_gdt` as they are completely empty.
- Drop `nom_etq2`, and `nom_etq3` because they duplicate the information in `nom`.
- Drop `population`, `densite_pop_km2`, and `source_mil` as these represent fact information handled in population-related tables.

In [25]:
communes_df.head(3)

,code_adm,code_insee,nature,nom,nom_etq2,nom_etq3,code_mel,libelle_mel,code_ut,libelle_ut,...,code_dt,libelle_dt,code_gdt,libelle_gdt,surface_ha,surface_km2,population,densite_pop_km2,source_mil,geom
0,COM_F59005,59005,Commune,Allennes-les-Marais,Allennes-les-Marais,Allennes-\n-les-Marais,MEL,Métropole Européenne de Lille,UTLS,UT Lille / Seclin,...,NaN,NaN,NaN,NaN,554,5.54,3460,624.55,IGN BD_TOPO 2020(2019) + INSEE 2020(2017),MULTIPOLYGON (((2.9630895928204106 50.52802738...
1,COM_F59009,59009,Commune,Villeneuve-d'Ascq,Villeneuve-d'Ascq,Villeneuve-\n-d'Ascq,MEL,Métropole Européenne de Lille,UTRV,UT Roubaix / Villeneuve d'Ascq,...,NaN,NaN,NaN,NaN,2748,27.48,63408,2307.42,IGN BD_TOPO 2020(2019) + INSEE 2020(2017),MULTIPOLYGON (((3.1793022723575555 50.60930052...
2,COM_F59011,59011,Commune,Annœullin,Annœullin,Annœullin,MEL,Métropole Européenne de Lille,UTLS,UT Lille / Seclin,...,NaN,NaN,NaN,NaN,898,8.98,10428,1161.25,IGN BD_TOPO 2020(2019) + INSEE 2020(2017),MULTIPOLYGON (((2.8954464720510598 50.53189944...


In [26]:
communes_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 97 entries, 0 to 96
Data columns (total 30 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   code_adm                        97 non-null     object 
 1   code_insee                      97 non-null     int64  
 2   nature                          97 non-null     object 
 3   nom                             97 non-null     object 
 4   nom_etq2                        97 non-null     object 
 5   nom_etq3                        97 non-null     object 
 6   code_mel                        94 non-null     object 
 7   libelle_mel                     94 non-null     object 
 8   code_ut                         97 non-null     object 
 9   libelle_ut                      97 non-null     object 
 10  code_territoire_gouvernance     97 non-null     object 
 11  libelle_territoire_gouvernance  97 non-null     object 
 12  code_deveco                     92 non

In [27]:
communes_df.describe(include = 'all')

,code_adm,code_insee,nature,nom,nom_etq2,nom_etq3,code_mel,libelle_mel,code_ut,libelle_ut,...,code_dt,libelle_dt,code_gdt,libelle_gdt,surface_ha,surface_km2,population,densite_pop_km2,source_mil,geom
count,97,97.000000,97,97,97,97,94,94,97,97,...,8,8,0.0,0.0,97.000000,97.000000,97.000000,97.000000,97,97
unique,97,NaN,3,97,97,97,1,1,4,4,...,1,1,NaN,NaN,NaN,NaN,NaN,NaN,1,97
top,COM_F59005,NaN,Commune,Allennes-les-Marais,Allennes-les-Marais,Allennes-\n-les-Marais,MEL,Métropole Européenne de Lille,UTML,UT Marcq-en-Baroeul / La Bassée,...,DT,Délégation Territoriale d'Armentières,NaN,NaN,NaN,NaN,NaN,NaN,IGN BD_TOPO 2020(2019) + INSEE 2020(2017),MULTIPOLYGON (((2.9630895928204106 50.52802738...
freq,1,NaN,94,1,1,1,94,94,35,35,...,8,8,NaN,NaN,NaN,NaN,NaN,NaN,97,1
mean,NaN,59348.989691,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,694.474227,6.944742,12068.350515,1661.748351,NaN,NaN
std,NaN,193.583073,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,481.233335,4.812333,24165.061283,2027.649688,NaN,NaN
min,NaN,59005.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,18.000000,0.180000,239.000000,57.040000,NaN,NaN
25%,NaN,59201.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,327.000000,3.270000,1769.000000,420.950000,NaN,NaN
50%,NaN,59346.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,554.000000,5.540000,4712.000000,741.050000,NaN,NaN
75%,NaN,59512.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,968.000000,9.680000,12608.000000,2139.380000,NaN,NaN


In [28]:
communes_df.nunique()

code_adm                          97
code_insee                        97
nature                             3
nom                               97
nom_etq2                          97
nom_etq3                          97
code_mel                           1
libelle_mel                        1
code_ut                            4
libelle_ut                         4
code_territoire_gouvernance        8
libelle_territoire_gouvernance     8
code_deveco                        5
libelle_deveco                     5
code_enm                           4
libelle_enm                        4
code_m_locale                      8
libelle_m_locale                   8
code_gardo                         1
libelle_gardo                      1
code_dt                            1
libelle_dt                         1
code_gdt                           0
libelle_gdt                        0
surface_ha                        95
surface_km2                       95
population                        97
d

## 4. Preliminary Observations

- From the `velos_df` **two dimension tables** will be separated: `dim_calendrier` and `dim_capteur`. Third **dimension table** will be created directly from `communes_df` as `dim_commune`.

- Two fact tables have been identified: `fact_velo` from `velos_df` and `fact_population` from `populations_df`. Both can be related to the `dim_commune` dimension table using its primary key `code_isnee`.

- All tables require **data cleaning and standardisation**, including handling missing values, correcting data types, and resolving string inconsistencies.

## 5. Next Steps

Based on the initial data exploration and profiling, the following steps will be performed in the next phase of the ETL project:

1. **Draft Star Schema**
   - Create a preliminary **star schema** diagram showing:
     - Fact tables (`fact_velo`, `fact_population`)
     - Connected dimension tables (`dim_commune`, `dim_calendrier`, `dim_capteur`)
   - Use this schema as a blueprint for the **PostgreSQL database** structure.

2. **Database Implementation**
   - Create empty PostgreSQL tables according to the draft schema.
   - Define primary keys, constraints, and column types.
  
3. **ETL Pipeline (Talend)**
   Implement data extraction, transformation, and loading from CSVs:

    - Create a calendar and a sensor tables.
    - Handle missing values in key columns using appropriate strategies (imputation).
    - Map and confirm key relationships:
        - `fact_velo.code_insee` → `dim_commune.code_insee`
        - `fact_population.code_insee` → `dim_commune.code_insee`
    - Finalize and enforce primary keys:
        - `dim_commune`: `code_isnee`
        - `dim_calendrier`: `calendrier_id`
        - `dim_capteur: capteur_id`
   - Remove redundant and constant fields (e.g., `epci`, `scot`, `nom`)
     

5. **Future Considerations**
   - Integrate `fact_population` with  `dim_calendrier` eshaping the population dataset into a long format.
   - Schedule automated ETL execution (e.g. cron-based orchestration).
   - Connect the PostgreSQL database wit Power BI to build an interactive dashboard.
   - Enrich the data model with contextual datasets (e.g., schools, parks, hospitals) to improve analytical depth.